# `BAOReconstructor` — interactive walkthrough

This example drives `baorecon`'s high-level `BAOReconstructor` on a **fully
self-contained mock** (no external files, no `pyrecon`) and shows how to reach
inside the object during an interactive session:

1. build a clustered mock (lognormal density + Poisson sampling) and a uniform random catalogue
2. construct a `BAOReconstructor` and read back the resolved geometry
3. inspect the **overdensity field** `delta_on_mesh`
4. inspect the **solver**: the potential $\phi$ and the displacement field $\Psi$ on the mesh
5. run the full reconstruction and look at the per-tracer displacements
6. see the real-space / RSD split of the displacement
7. compare galaxy positions **before vs after** reconstruction

Everything runs on the CPU in a couple of minutes at `nmesh=128`.

## 0. Thread control (must run first)

The number of threads used by numba, BLAS and the FFTs is read from environment
variables **at import time**, so these must be set *before* `numpy` / `numba` /
`baorecon` are imported. See the **Multithreading** section of the repository
README for the full story.

In [ ]:
import os

THREADS = "8"                        # match this to the cores you were allocated
os.environ["NUMBA_NUM_THREADS"]    = THREADS   # numba JIT kernels (MAS, interp, divergence, ...)
os.environ["OMP_NUM_THREADS"]      = THREADS   # OpenMP / BLAS
os.environ["OPENBLAS_NUM_THREADS"] = THREADS
os.environ["MKL_NUM_THREADS"]      = THREADS
os.environ["NUMEXPR_NUM_THREADS"]  = THREADS
# os.environ["BAORECON_FFT"]         = "pyfftw"  # optional low-memory in-place CPU FFT
# os.environ["BAORECON_FFT_THREADS"] = THREADS

In [ ]:
import sys

import numpy as np
import matplotlib.pyplot as plt

try:
    from baorecon import BAOReconstructor
except ModuleNotFoundError:
    # Running from a source checkout without installing: add the repo root.
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
    from baorecon import BAOReconstructor

from baorecon.utils.mock_generator import generate_gaussian_map, poisson_sample_from_map

%matplotlib inline

## 1. Build a mock catalogue

We draw a Gaussian field from a smooth toy power spectrum, apply a **lognormal
transform** (so that $1+\delta > 0$ everywhere), and **Poisson-sample** galaxies
from it. The randoms are uniform in the cubic box — the natural selection
function for a periodic mock.

In [ ]:
np.random.seed(42)

BOXSIZE = 1000.0            # Mpc/h, cubic periodic box
NCELL   = 64               # grid used only to *generate* the field
SPACING = BOXSIZE / NCELL

def pk_model(k):
    """A smooth, broad toy power spectrum (finite everywhere)."""
    k  = np.asarray(k, dtype=float)
    kk = np.where(k > 0, k, 1e-6)
    P  = 2.5e3 * (kk / 0.05) / (1.0 + (kk / 0.15) ** 3)
    return np.where(k > 0, P, 0.0)

# Gaussian field -> lognormal transform (guarantees 1 + delta > 0)
_, delta_g = generate_gaussian_map(pk_model, BOXSIZE, SPACING)
delta_mock = np.exp(delta_g - np.var(delta_g) / 2.0) - 1.0

# Poisson-sample galaxies; randoms uniform in the box (3x the data)
data_pos   = poisson_sample_from_map(delta_mock, BOXSIZE, SPACING,
                                     N_objects=int(3e5), seed=1)
rng        = np.random.default_rng(7)
random_pos = rng.uniform(0.0, BOXSIZE, size=(3 * len(data_pos), 3))

print(f"data:    {data_pos.shape[0]:,} galaxies")
print(f"randoms: {random_pos.shape[0]:,} points")

In [ ]:
# A thin z-slice of the input catalogue shows the clustering we will reconstruct
sl = data_pos[:, 2] < 60.0
plt.figure(figsize=(5.5, 5))
plt.scatter(data_pos[sl, 0], data_pos[sl, 1], s=2, alpha=0.4)
plt.xlabel("X [Mpc/h]"); plt.ylabel("Y [Mpc/h]")
plt.title("Input galaxies (z < 60 Mpc/h slice)")
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

## 2. Construct the `BAOReconstructor`

All the physics/geometry choices are passed once to the constructor. Nothing is
computed yet — the density field and the solver are built lazily on first access.

> **Note.** Our mock is a plain real-space density field (no RSD imprinted). We
> still run in `RedshiftSpace` below purely to demonstrate how the reconstructor
> splits the displacement into a real-space part and a line-of-sight RSD part.

In [ ]:
BIAS      = 1.8               # linear galaxy bias
GROWTH    = 0.8              # growth rate f
RSD_SPACE = "RedshiftSpace"  # "RealSpace" or "RedshiftSpace"

NMESH     = 128              # reconstruction mesh (128^3)
LOS       = "z"              # 'x'/'y'/'z' plane-parallel, or None for local (radial)
R_SM      = 15.0             # Gaussian smoothing radius [Mpc/h]
MAS       = "CIC"            # mass-assignment scheme ("CIC" or "TSC")
SOLVER    = "ifft"           # "ifft" (Burden iFFT) or "multigrid"

In [ ]:
recon = BAOReconstructor(
    data_pos=data_pos,
    random_pos=random_pos,
    RSDspace=RSD_SPACE,
    nmesh=NMESH,
    boxsize=BOXSIZE,
    los=LOS,
    f=GROWTH,
    bias=BIAS,
    R_sm=R_SM,
    padding=0.0,        # the box already encloses the catalogue
    pbc=True,           # periodic cubic mock
    MAS=MAS,
    solver_type=SOLVER,
    device="cpu",       # set "gpu" if CuPy + a CUDA GPU are available
)

In [ ]:
recon.print_info()

## 3. Inspect the resolved geometry

`nmesh`/`boxsize`/`boxcentre` are resolved in `__init__` (they can also be
derived from a target `cellsize`). They are exposed as read-only properties.

In [ ]:
print("nmesh     :", recon.nmesh)
print("boxsize   :", recon.boxsize, "Mpc/h")
print("cell_size :", recon.cell_size, "Mpc/h")
print("boxcentre :", recon.boxcentre)
print("mesh      :", recon.mesh)
print("MAS / los :", recon.MAS, "/", recon.los)
print("f / bias  :", recon.f, "/", recon.bias)
print("dtype     :", recon.dtype)

## 4. The overdensity field `delta_on_mesh`

`delta_on_mesh` is a lazy property: the first access runs the mass assignment and
builds the smoothed overdensity field $\delta(\mathbf{x})$ on the mesh.

In [ ]:
delta = recon.delta_on_mesh
print("delta_on_mesh:", type(delta).__name__, delta.shape, delta.dtype)
print(f"mean={delta.mean():.3e}   std={delta.std():.3f}   "
      f"min={delta.min():.2f}   max={delta.max():.2f}")

In [ ]:
N  = int(np.asarray(recon.nmesh)[0])
z0 = N // 2

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
im0 = axs[0].imshow(delta[:, :, z0].T, origin="lower", cmap="viridis")
axs[0].set_title(f"$\\delta$ slice (z index = {z0})")
fig.colorbar(im0, ax=axs[0], label=r"$\delta$")

im1 = axs[1].imshow(delta.sum(axis=2).T, origin="lower", cmap="viridis")
axs[1].set_title(r"$\delta$ projected along z")
fig.colorbar(im1, ax=axs[1], label=r"$\sum_z \delta$")
for ax in axs:
    ax.set_xlabel("X"); ax.set_ylabel("Y")
plt.tight_layout(); plt.show()

## 5. The solver: potential $\phi$ and displacement $\Psi$

`recon.solver` is built lazily on first access. Its `potential` and
`displacement` are themselves lazy — touching them triggers the Poisson solve.
The displacement is $\Psi = -\nabla\phi$, returned as a contiguous
`(N, N, N, 3)` field.

In [ ]:
solver = recon.solver
print("solver:", type(solver).__name__)

phi = recon.solver.potential       # scalar potential on the mesh, (N, N, N)
psi = recon.solver.displacement    # displacement field Psi, (N, N, N, 3)
print("potential   :", np.asarray(phi).shape)
print("displacement:", np.asarray(psi).shape, psi.dtype)

In [ ]:
step = 4
xs = np.arange(0, N, step)
Xg, Yg = np.meshgrid(xs, xs)           # default 'xy' indexing
U = psi[::step, ::step, z0, 0].T       # Psi_x on the z0 slice
V = psi[::step, ::step, z0, 1].T       # Psi_y on the z0 slice

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
im0 = axs[0].imshow(phi[:, :, z0].T, origin="lower", cmap="viridis")
axs[0].set_title(r"potential $\phi$ + displacement $\Psi$")
fig.colorbar(im0, ax=axs[0], label=r"$\phi$")

vmax = np.abs(delta[:, :, z0]).max()
im1 = axs[1].imshow(delta[:, :, z0].T, origin="lower", cmap="bwr",
                    vmin=-vmax, vmax=vmax)
axs[1].set_title(r"density $\delta$ + displacement $\Psi$")
fig.colorbar(im1, ax=axs[1], label=r"$\delta$")

for ax in axs:
    ax.quiver(Xg, Yg, U, V, color="k", scale=120, width=0.003)
    ax.set_xlabel("X"); ax.set_ylabel("Y")
plt.tight_layout(); plt.show()

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(18, 4))
for i, comp in enumerate(["X", "Y", "Z"]):
    axs[i].hist(psi[..., i].ravel(), bins=60, color="skyblue", edgecolor="k")
    axs[i].set_title(f"{comp} component")
    axs[i].set_xlabel(rf"$\Psi_{comp}$ [Mpc/h]")
    axs[i].text(0.95, 0.95,
                f"mean {psi[..., i].mean():.2e}\nstd  {psi[..., i].std():.2e}",
                transform=axs[i].transAxes, ha="right", va="top",
                bbox=dict(boxstyle="round", fc="white", alpha=0.8))
mag = np.linalg.norm(psi, axis=-1)
axs[3].hist(mag.ravel(), bins=60, color="salmon", edgecolor="k")
axs[3].set_title("magnitude"); axs[3].set_xlabel(r"$|\Psi|$ [Mpc/h]")
plt.suptitle("Displacement field on the mesh")
plt.tight_layout(); plt.show()

## 6. Run the full reconstruction

`run_reconstruction()` interpolates the mesh displacement onto the tracers and
returns the shifted **data** and **random** catalogues.

In [ ]:
data_rec, random_rec = recon.run_reconstruction()
print("reconstructed galaxies:", data_rec.shape)
print("reconstructed randoms :", random_rec.shape)

## 7. Per-tracer displacements & the RSD split

You can also call the interpolation machinery directly. `_interpolate_displacement`
reads $\Psi$ at arbitrary positions; `_get_rsd_displacement` returns the
line-of-sight (RSD) part, which is non-zero only in `RedshiftSpace`.

In [ ]:
psi_real = recon._interpolate_displacement(recon.data_pos)   # (N, 3) real-space part
psi_rsd  = recon._get_rsd_displacement(psi_real, recon.data_pos)
psi_tot  = psi_real + psi_rsd

fig, axs = plt.subplots(1, 4, figsize=(18, 4))
bins = np.linspace(np.min(psi_tot), np.max(psi_tot), 60)
for i, comp in enumerate(["X", "Y", "Z"]):
    axs[i].hist(psi_real[:, i], bins=bins, density=True, alpha=0.5, label="real space")
    axs[i].hist(psi_tot[:, i],  bins=bins, density=True, histtype="step", lw=2, label="real + RSD")
    axs[i].set_title(f"{comp} component"); axs[i].set_xlabel(rf"$\Psi_{comp}$ [Mpc/h]")
    axs[i].legend()
axs[3].hist(np.linalg.norm(psi_real, axis=1), bins=60, density=True, alpha=0.5, label="real space")
axs[3].hist(np.linalg.norm(psi_tot,  axis=1), bins=60, density=True, histtype="step", lw=2, label="real + RSD")
axs[3].set_title("magnitude"); axs[3].set_xlabel(r"$|\Psi|$ [Mpc/h]"); axs[3].legend()
plt.suptitle("Per-tracer displacement (interpolated from the mesh)")
plt.tight_layout(); plt.show()

## 8. Positions before vs after

Finally, the payoff: galaxies get pushed back along the reconstructed
displacement. We convert to the box frame (a `baorecon` helper) and quiver a
thin corner sub-volume so the arrows are legible.

In [ ]:
from baorecon.utils.formatters import survey_to_box_frame

min_corner = recon._density_manager.min_corner
data_box = survey_to_box_frame(recon.data_pos, min_corner, recon.boxsize, pbc=recon.pbc)
rec_box  = survey_to_box_frame(data_rec,        min_corner, recon.boxsize, pbc=recon.pbc)

depth, width = 40.0, 200.0
m = (data_box[:, 0] < width) & (data_box[:, 1] < width) & (data_box[:, 2] < depth)

plt.figure(figsize=(7, 7))
plt.scatter(data_box[m, 0], data_box[m, 1], s=14, color="black", label="original")
plt.scatter(rec_box[m, 0],  rec_box[m, 1],  s=26, facecolors="none",
            edgecolors="tab:blue", label="reconstructed")
plt.quiver(data_box[m, 0], data_box[m, 1],
           rec_box[m, 0] - data_box[m, 0], rec_box[m, 1] - data_box[m, 1],
           angles="xy", scale_units="xy", scale=1.0, color="tab:blue",
           width=0.004, alpha=0.8)
plt.xlabel("X [Mpc/h]"); plt.ylabel("Y [Mpc/h]")
plt.title(f"Galaxy shift under reconstruction (z < {depth:.0f} Mpc/h)")
plt.gca().set_aspect("equal"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
shift = np.linalg.norm(data_rec - recon.data_pos, axis=1)
plt.figure(figsize=(6, 4))
plt.hist(shift, bins=60, color="tab:green", edgecolor="k")
plt.xlabel("|shift| applied to galaxies [Mpc/h]"); plt.ylabel("count")
plt.title(f"Reconstruction shift magnitude (median {np.median(shift):.2f} Mpc/h)")
plt.tight_layout(); plt.show()

## Recap & next steps

You built a mock, constructed a `BAOReconstructor`, reached into its lazy
internals (`delta_on_mesh`, `solver.potential`, `solver.displacement`), ran the
reconstruction, and split the per-tracer displacement into a real-space and an
RSD part.

**Handy handles on the object**

| Attribute / method | What it gives you |
|---|---|
| `recon.nmesh`, `recon.boxsize`, `recon.cell_size`, `recon.boxcentre` | resolved geometry |
| `recon.mesh` | the `Mesh` object |
| `recon.delta_on_mesh` | overdensity field ($N^3$) |
| `recon.solver` | the displacement solver (lazy) |
| `recon.solver.potential` / `.displacement` | $\phi$ and $\Psi$ on the mesh |
| `recon.run_reconstruction()` | shifted data & randoms |
| `recon._interpolate_displacement(pos)` | $\Psi$ interpolated at positions |
| `recon._get_rsd_displacement(psi, pos)` | the LOS/RSD part of $\Psi$ |
| `recon._density_manager.min_corner` | survey → box frame offset |

**Try next**

- Switch `SOLVER = "multigrid"`, or set `LOS = None` for a local (radial) line of sight, and re-run.
- Set `device="gpu"` if you have CuPy and a CUDA GPU.
- For a one-call API without the object, use `reconstruct_positions`:

```python
from baorecon import reconstruct_positions
shifted_data, shifted_randoms = reconstruct_positions(
    data_pos, random_pos, f=GROWTH, bias=BIAS,
    nmesh=NMESH, smoothing=R_SM, los="z", device="cpu",
)
```

- For survey catalogues (FITS/Parquet), use the YAML-driven pipeline in `baorecon/pipeline/`.
- See the repository README **Multithreading** section for the environment variables set in the first cell.